In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)

import os
from PIL import Image

In [2]:
root_path = "/Users/jwatts/astrophotography/Processed/"

In [3]:
def create_image_dataframe(root_path):
    data = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.png'):
                image_path = os.path.join(root, file)
                directory_name = os.path.basename(root).lower().replace(' ', '')
                data.append([image_path, directory_name])
    return pd.DataFrame(data, columns=['image_name', 'objectId'])

def remap_filenames(df):
    remap_dict = {
    }

    def remap_filename(filename):
        return remap_dict.get(filename, filename)

    df['objectId'] = df['objectId'].apply(remap_filename)
    return df

def process_images(df, output_path):
    deep_sky_objects = []

    for index, row in df.iterrows():
        image_path = row['image_name']
        output_filename = row['objectId']
        #name = row['name']
        
        with Image.open(image_path) as img:
            # Save the processed image in WEBP format in the output folder
            optimized_image_path = os.path.join(output_path, f'{output_filename}.webp')
            
            # Resize the image if the 'resize' column exists and the resize height is specified and necessary
            #if 'resize' in row and row['resize'] > 0 and img.height > row['resize']:
            #    aspect_ratio = img.width / img.height
            #    new_width = int(row['resize'] * aspect_ratio)
            #    img = img.resize((new_width, row['resize']), Image.Resampling.LANCZOS)

            img.save(optimized_image_path, 'WEBP')
            
            # Add the object to the list
            #deep_sky_objects.append({'name': name, 'image': f'/images/{output_filename}.webp'})

    # Generate the JavaScript array code
    #js_array_code = f'const deepSkyObjects = {deep_sky_objects};'

    # Print or return the code
    #print(js_array_code)
    #return js_array_code


In [ ]:
df = create_image_dataframe(root_path)
df = remap_filenames(df)

# Create the 'optimized' directory if it doesn't exist
output_path = "/Users/jwatts/astrophotography/Cloud"

if not os.path.exists(output_path):
    os.makedirs(output_path)

process_images(df, output_path)

In [ ]:
df['crop'] = 0

df.loc[df['objectId'] == "m1", 'crop'] = 1300  # Crab Nebula
df.loc[df['objectId'] == "m13", 'crop'] = 1750  # Hercules Globular Cluster
df.loc[df['objectId'] == "m51", 'crop'] = 1080  # Whirlpool Galaxy
df.loc[df['objectId'] == "m63", 'crop'] = 1750  # Whirlpool Galaxy
df.loc[df['objectId'] == "m82", 'crop'] = 1750  
df.loc[df['objectId'] == "m97", 'crop'] = 1080  # Owl nebula
df.loc[df['objectId'] == "m101", 'crop'] = 1500  # Pinwheel Galaxy
df.loc[df['objectId'] == "m104", 'crop'] = 1080  # Sombrero Galaxy

imagesJson = output_path + '/images.json' 
df.drop(columns=['image_name']).to_json(imagesJson, orient='records', lines=False)

df[0:100]